In [17]:
# =========================
# CELL 1: Imports + Config
# =========================

import os
import math
import random
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from shapely import wkt
from shapely.geometry import Polygon, LineString
from shapely import affinity

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

# -------------------------
# Reproducibility
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------------------------
# Device
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# -------------------------
# Paths
# -------------------------
TSV_PATH = "/raid/ruban/data/parks.tsv"   # change if needed

OUT_DIR = Path("/raid/ruban/hpmlproj/embedding")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Data / Model Config
# -------------------------
MAX_POLYGONS = 50000        # start smaller if testing
NUM_POINTS = 128            # fixed boundary points per polygon
EMBED_DIM = 128             # output embedding size

BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-4

# Jaccard thresholds for training pairs/triplets
POSITIVE_JACCARD_TH = 0.70
NEGATIVE_JACCARD_TH = 0.30

print("Output dir:", OUT_DIR)

Using device: cuda
Output dir: /raid/ruban/hpmlproj/embedding


In [18]:
# ==========================================
# CELL 2: Load parks.tsv -> Polygon objects
# ==========================================

def read_parks_tsv(file_path, max_polygons=None):
    polys = []
    poly_ids = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Reading TSV"):
            try:
                parts = line.strip().split("\t")

                if len(parts) < 2:
                    continue

                poly_id = parts[0]
                geom_txt = parts[1]

                geom = wkt.loads(geom_txt)

                # Your parks file often stores polygon boundaries as LINESTRING
                if geom.geom_type == "LineString":
                    coords = list(geom.coords)

                    if len(coords) >= 3:
                        poly = Polygon(coords)

                    else:
                        continue

                elif geom.geom_type == "Polygon":
                    poly = geom

                else:
                    continue

                # Clean invalid shapes
                if not poly.is_valid:
                    poly = poly.buffer(0)

                if poly.is_empty:
                    continue

                if poly.area <= 0:
                    continue

                polys.append(poly)
                poly_ids.append(poly_id)

                if max_polygons and len(polys) >= max_polygons:
                    break

            except Exception:
                continue

    print("Loaded polygons:", len(polys))
    return poly_ids, polys


# Run loader
poly_ids, polygons = read_parks_tsv(TSV_PATH, max_polygons=MAX_POLYGONS)

print("Sample ID:", poly_ids[0])
print("Sample Area:", polygons[0].area)
print("Sample Type:", polygons[0].geom_type)

Reading TSV: 0it [00:00, ?it/s]

Loaded polygons: 50000
Sample ID: 4061698
Sample Area: 3.604821063999418e-05
Sample Type: Polygon


In [19]:
# ==========================================
# CELL 3: Polygon -> Fixed 128 boundary points
# ==========================================

def normalize_polygon(poly):
    """
    Center polygon at origin and scale into unit box.
    """
    c = poly.centroid

    # move centroid to origin
    poly = affinity.translate(poly, xoff=-c.x, yoff=-c.y)

    minx, miny, maxx, maxy = poly.bounds
    width = maxx - minx
    height = maxy - miny

    scale = max(width, height)

    if scale > 0:
        poly = affinity.scale(poly, xfact=1/scale, yfact=1/scale, origin=(0, 0))

    return poly


def sample_boundary_points(poly, num_points=128):
    """
    Sample equally spaced points along polygon exterior boundary.
    Returns shape: (num_points, 2)
    """
    poly = normalize_polygon(poly)

    ring = LineString(poly.exterior.coords)

    total_len = ring.length

    pts = []
    for i in range(num_points):
        d = (i / num_points) * total_len
        p = ring.interpolate(d)
        pts.append([p.x, p.y])

    return np.array(pts, dtype=np.float32)


# Test on one polygon
sample_pts = sample_boundary_points(polygons[0], NUM_POINTS)

print("Shape:", sample_pts.shape)
print(sample_pts[:5])

Shape: (128, 2)
[[-0.52441466  0.1412817 ]
 [-0.5192463   0.12115893]
 [-0.50373393  0.11510491]
 [-0.4829859   0.11618163]
 [-0.46223795  0.11725836]]


In [20]:
# ==========================================
# CELL 4: Build Dataset Tensor + Cache
# ==========================================

CACHE_X = OUT_DIR / f"polygon_points_{len(polygons)}_{NUM_POINTS}.npy"

if CACHE_X.exists():
    print("Loading cached tensor...")
    X = np.load(CACHE_X)

else:
    print("Generating boundary tensors...")

    X = np.zeros((len(polygons), NUM_POINTS, 2), dtype=np.float32)

    for i, poly in enumerate(tqdm(polygons)):
        try:
            X[i] = sample_boundary_points(poly, NUM_POINTS)
        except Exception:
            # fallback = zeros if broken polygon
            X[i] = np.zeros((NUM_POINTS, 2), dtype=np.float32)

    np.save(CACHE_X, X)
    print("Saved cache:", CACHE_X)

print("Tensor shape:", X.shape)
print("Memory MB:", round(X.nbytes / 1024 / 1024, 2))

Generating boundary tensors...


  0%|          | 0/50000 [00:00<?, ?it/s]

Saved cache: /raid/ruban/hpmlproj/embedding/polygon_points_50000_128.npy
Tensor shape: (50000, 128, 2)
Memory MB: 48.83


In [21]:
# ==========================================
# CELL 5: Load GT Similarity Neighbor Files
# ==========================================

import glob

GT_DIR = "/raid/ruban/groundtruth/pk-query-50k"   # change if needed

def load_similarity_maps(gt_dir, max_queries=None):
    gt = {}

    files = sorted(glob.glob(os.path.join(gt_dir, "similarityMap_*")))

    print("GT files found:", len(files))

    for fp in tqdm(files, desc="Loading GT"):
        with open(fp, "r") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                parts = [x.strip() for x in line.split(",")]

                qid = int(parts[0])
                nbrs = [int(x) for x in parts[1:] if x != ""]

                gt[qid] = nbrs

                if max_queries and len(gt) >= max_queries:
                    return gt

    return gt


# Load GT
gt_map = load_similarity_maps(GT_DIR)

print("Queries loaded:", len(gt_map))

# Show sample
sample_q = next(iter(gt_map))
print("Sample Query:", sample_q)
print("Top neighbors:", gt_map[sample_q][:10])

GT files found: 60


Loading GT:   0%|          | 0/60 [00:00<?, ?it/s]

Queries loaded: 10000
Sample Query: 40000
Top neighbors: [26769, 35012, 12304, 39137, 28042, 27638, 23490, 30296, 29411, 14399]


In [22]:
# ==========================================
# CELL 6: Build Training Triplets
# ==========================================

def build_triplets(gt_map, db_size, top_pos=10, negatives_per_query=3):
    triplets = []

    all_ids = set(range(db_size))

    for qid, nbrs in tqdm(gt_map.items(), desc="Building triplets"):

        if qid >= db_size:
            continue

        positives = [x for x in nbrs[:top_pos] if x < db_size]

        if not positives:
            continue

        pos_set = set(positives)
        forbidden = pos_set | {qid}

        candidates_neg = list(all_ids - forbidden)

        for p in positives:
            for _ in range(negatives_per_query):
                n = random.choice(candidates_neg)
                triplets.append((qid, p, n))

    return triplets


triplets = build_triplets(
    gt_map=gt_map,
    db_size=len(X),
    top_pos=10,
    negatives_per_query=3
)

print("Total triplets:", len(triplets))
print("Sample:", triplets[:5])

Building triplets:   0%|          | 0/10000 [00:00<?, ?it/s]

Total triplets: 272220
Sample: [(40000, 26769, 41916), (40000, 26769, 7296), (40000, 26769, 1639), (40000, 35012, 48609), (40000, 35012, 18026)]


In [27]:
# ==========================================
# CELL 7: PyTorch Triplet Dataset
# ==========================================

class PolygonTripletDataset(Dataset):
    def __init__(self, X, triplets):
        """
        X = polygon tensor (N,128,2)
        triplets = list of (anchor,pos,neg)
        """
        self.X = X
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a, p, n = self.triplets[idx]

        xa = torch.tensor(self.X[a], dtype=torch.float32)
        xp = torch.tensor(self.X[p], dtype=torch.float32)
        xn = torch.tensor(self.X[n], dtype=torch.float32)

        return xa, xp, xn


# Build dataset
train_ds = PolygonTripletDataset(X, triplets)

# DataLoader
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

# Test one batch
xa, xp, xn = next(iter(train_loader))

print("Anchor batch:", xa.shape)
print("Positive batch:", xp.shape)
print("Negative batch:", xn.shape)

Anchor batch: torch.Size([128, 128, 2])
Positive batch: torch.Size([128, 128, 2])
Negative batch: torch.Size([128, 128, 2])


In [28]:
# ==========================================
# CELL 8: Polygon Encoder (1D CNN)
# ==========================================

class PolygonCNNEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(2, 64, kernel_size=5, padding=2),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.ReLU(),

            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.ReLU()
        )

        self.fc = nn.Linear(256, embed_dim)

    def forward(self, x):
        """
        x shape: (B,128,2)
        """

        # convert to (B,2,128)
        x = x.permute(0, 2, 1)

        feat = self.net(x)

        # global max pool over sequence
        feat = torch.max(feat, dim=2).values

        emb = self.fc(feat)

        # normalize embeddings
        emb = F.normalize(emb, p=2, dim=1)

        return emb


# Create model
model = PolygonCNNEncoder(embed_dim=EMBED_DIM).to(DEVICE)

# Test forward pass
with torch.no_grad():
    test_emb = model(xa.to(DEVICE))

print("Embedding shape:", test_emb.shape)
print("Norm first vector:", test_emb[0].norm().item())

Embedding shape: torch.Size([128, 128])
Norm first vector: 1.0


In [29]:
# ==========================================
# CELL 9: Train with Triplet Loss
# ==========================================

criterion = nn.TripletMarginLoss(margin=0.30, p=2)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    total_batches = 0

    pbar = tqdm(loader)

    for xa, xp, xn in pbar:

        xa = xa.to(DEVICE, non_blocking=True)
        xp = xp.to(DEVICE, non_blocking=True)
        xn = xn.to(DEVICE, non_blocking=True)

        ea = model(xa)
        ep = model(xp)
        en = model(xn)

        loss = criterion(ea, ep, en)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

        avg = total_loss / total_batches
        pbar.set_description(f"loss={avg:.4f}")

    return total_loss / total_batches


# Run first epoch
loss_val = train_one_epoch(model, train_loader)

print("Epoch Loss:", round(loss_val, 4))

  0%|          | 0/2127 [00:00<?, ?it/s]

Epoch Loss: 0.2994


In [30]:
# ==========================================
# CELL 9B: Train Multiple Epochs
# ==========================================

history = []

for epoch in range(1, 6):

    t0 = time.time()

    loss_val = train_one_epoch(model, train_loader)

    dt = time.time() - t0

    history.append(loss_val)

    print(f"Epoch {epoch} | Loss={loss_val:.4f} | Time={dt:.1f}s")

  0%|          | 0/2127 [00:00<?, ?it/s]

Epoch 1 | Loss=0.2987 | Time=16.6s


  0%|          | 0/2127 [00:00<?, ?it/s]

Epoch 2 | Loss=0.2982 | Time=16.6s


  0%|          | 0/2127 [00:00<?, ?it/s]

Epoch 3 | Loss=0.2974 | Time=16.4s


  0%|          | 0/2127 [00:00<?, ?it/s]

Epoch 4 | Loss=0.2969 | Time=16.5s


  0%|          | 0/2127 [00:00<?, ?it/s]

Epoch 5 | Loss=0.2961 | Time=16.6s


In [31]:
# ==========================================
# CELL A: Build (polyA, polyB, target_score)
# ==========================================

def rank_to_score(rank_idx):
    """
    Convert GT rank -> soft similarity target
    rank_idx = 0 means nearest neighbor
    """
    if rank_idx == 0:
        return 1.00
    elif rank_idx < 3:
        return 0.95
    elif rank_idx < 5:
        return 0.90
    elif rank_idx < 10:
        return 0.80
    elif rank_idx < 20:
        return 0.70
    else:
        return 0.60


def build_pair_dataset(gt_map, db_size, negatives_per_query=5):
    pairs = []

    all_ids = list(range(db_size))

    for qid, nbrs in tqdm(gt_map.items(), desc="Building pairs"):

        if qid >= db_size:
            continue

        # ---------------------------
        # Positive ranked pairs
        # ---------------------------
        for rank_idx, nb in enumerate(nbrs):

            if nb >= db_size:
                continue

            score = rank_to_score(rank_idx)

            pairs.append((qid, nb, score))

        # ---------------------------
        # Random negatives
        # ---------------------------
        forbidden = set(nbrs) | {qid}

        for _ in range(negatives_per_query):
            neg = random.choice(all_ids)

            while neg in forbidden:
                neg = random.choice(all_ids)

            pairs.append((qid, neg, 0.0))

    return pairs


pairs = build_pair_dataset(gt_map, len(X), negatives_per_query=5)

print("Total pairs:", len(pairs))
print("Sample pairs:", pairs[:10])

Building pairs:   0%|          | 0/10000 [00:00<?, ?it/s]

Total pairs: 10087825
Sample pairs: [(40000, 26769, 1.0), (40000, 35012, 0.95), (40000, 12304, 0.95), (40000, 39137, 0.9), (40000, 28042, 0.9), (40000, 27638, 0.8), (40000, 23490, 0.8), (40000, 30296, 0.8), (40000, 29411, 0.8), (40000, 14399, 0.8)]


In [32]:
# ==========================================
# CELL B: Pair Dataset + DataLoader
# ==========================================

MAX_TRAIN_PAIRS = 1_000_000   # fast first run

if len(pairs) > MAX_TRAIN_PAIRS:
    train_pairs = random.sample(pairs, MAX_TRAIN_PAIRS)
else:
    train_pairs = pairs

print("Using pairs:", len(train_pairs))


class PolygonPairDataset(Dataset):
    def __init__(self, X, pairs):
        self.X = X
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        a, b, y = self.pairs[idx]

        xa = torch.tensor(self.X[a], dtype=torch.float32)
        xb = torch.tensor(self.X[b], dtype=torch.float32)
        y  = torch.tensor(y, dtype=torch.float32)

        return xa, xb, y


pair_ds = PolygonPairDataset(X, train_pairs)

pair_loader = DataLoader(
    pair_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

# test batch
xa, xb, y = next(iter(pair_loader))

print("xa:", xa.shape)
print("xb:", xb.shape)
print("y :", y.shape)
print("targets sample:", y[:10])

Using pairs: 1000000
xa: torch.Size([256, 128, 2])
xb: torch.Size([256, 128, 2])
y : torch.Size([256])
targets sample: tensor([0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000,
        0.6000])


In [33]:
# ==========================================
# CELL C0: Balanced Sampling
# ==========================================

from collections import defaultdict

bins = defaultdict(list)

for row in pairs:
    _, _, score = row
    bins[score].append(row)

for k in sorted(bins.keys()):
    print(k, len(bins[k]))

TARGET_PER_BIN = 150_000

balanced_pairs = []

for score in sorted(bins.keys()):
    rows = bins[score]

    take = min(TARGET_PER_BIN, len(rows))

    balanced_pairs.extend(random.sample(rows, take))

random.shuffle(balanced_pairs)

print("Balanced pair count:", len(balanced_pairs))
print("First 20 scores:", [x[2] for x in balanced_pairs[:20]])

0.0 50000
0.6 9860179
0.7 86906
0.8 44666
0.9 18181
0.95 18476
1.0 9417
Balanced pair count: 377646
First 20 scores: [0.7, 0.6, 0.7, 0.6, 0.8, 0.0, 0.6, 1.0, 0.95, 0.7, 0.6, 0.0, 0.8, 0.6, 0.0, 0.6, 0.8, 0.7, 0.7, 0.6]


In [34]:
# ==========================================
# CELL C: Train Cosine -> Target Score
# ==========================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-4
)

criterion = nn.MSELoss()


def train_pair_epoch(model, loader):
    model.train()

    total_loss = 0.0
    count = 0

    pbar = tqdm(loader)

    for xa, xb, y in pbar:

        xa = xa.to(DEVICE, non_blocking=True)
        xb = xb.to(DEVICE, non_blocking=True)
        y  = y.to(DEVICE, non_blocking=True)

        ea = model(xa)
        eb = model(xb)

        pred = F.cosine_similarity(ea, eb)

        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

        avg = total_loss / count
        pbar.set_description(f"loss={avg:.4f}")

    return total_loss / count


history = []

for epoch in range(1, 6):
    t0 = time.time()

    loss_val = train_pair_epoch(model, pair_loader)

    dt = time.time() - t0

    history.append(loss_val)

    print(f"Epoch {epoch} | Loss={loss_val:.4f} | Time={dt:.1f}s")

  0%|          | 0/3907 [00:00<?, ?it/s]

Epoch 1 | Loss=0.0074 | Time=68.1s


  0%|          | 0/3907 [00:00<?, ?it/s]

Epoch 2 | Loss=0.0051 | Time=45.9s


  0%|          | 0/3907 [00:00<?, ?it/s]

Epoch 3 | Loss=0.0047 | Time=47.5s


  0%|          | 0/3907 [00:00<?, ?it/s]

Epoch 4 | Loss=0.0045 | Time=51.3s


  0%|          | 0/3907 [00:00<?, ?it/s]

Epoch 5 | Loss=0.0044 | Time=45.6s


In [35]:
# ==========================================
# CELL D: Generate Embeddings for All Polygons
# ==========================================

EMB_PATH = OUT_DIR / f"cnn_embeddings_{len(X)}_{EMBED_DIM}.npy"

def generate_embeddings(model, X, batch_size=512):
    model.eval()
    all_embs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(X), batch_size), desc="Generating embeddings"):
            xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            emb = model(xb)
            all_embs.append(emb.cpu().numpy())

    return np.vstack(all_embs).astype("float32")


embeddings = generate_embeddings(model, X, batch_size=512)

np.save(EMB_PATH, embeddings)

print("Embeddings shape:", embeddings.shape)
print("Saved to:", EMB_PATH)
print("First vector norm:", np.linalg.norm(embeddings[0]))

Generating embeddings:   0%|          | 0/98 [00:00<?, ?it/s]

Embeddings shape: (50000, 128)
Saved to: /raid/ruban/hpmlproj/embedding/cnn_embeddings_50000_128.npy
First vector norm: 0.99999994


In [36]:
# ==========================================
# CELL E: FAISS Recall@10 Evaluation
# ==========================================

import faiss

# embeddings already normalized -> cosine via inner product
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS index size:", index.ntotal)


def recall_at_k(gt_map, search_ids, k=10):
    recalls = []

    for qid in tqdm(search_ids, desc=f"Recall@{k}"):

        if qid >= len(embeddings):
            continue

        gt = gt_map.get(qid, [])
        if len(gt) == 0:
            continue

        gt_topk = set(gt[:k])

        qvec = embeddings[qid:qid+1]

        scores, idxs = index.search(qvec, k + 1)

        pred = [x for x in idxs[0] if x != qid][:k]

        hit = len(set(pred) & gt_topk)

        recalls.append(hit / k)

    return float(np.mean(recalls))


query_ids = sorted(gt_map.keys())

r10 = recall_at_k(gt_map, query_ids, k=10)
r50 = recall_at_k(gt_map, query_ids, k=50)

print("Recall@10 :", round(r10, 4))
print("Recall@50 :", round(r50, 4))

FAISS index size: 50000


Recall@10:   0%|          | 0/10000 [00:00<?, ?it/s]

Recall@50:   0%|          | 0/10000 [00:00<?, ?it/s]

Recall@10 : 0.0002
Recall@50 : 0.001


In [39]:
# ==========================================
# CELL F3: Center + Scale Jaccard
# ==========================================

def normalize_shape(poly):
    c = poly.centroid
    poly = affinity.translate(poly, xoff=-c.x, yoff=-c.y)

    minx, miny, maxx, maxy = poly.bounds
    scale = max(maxx - minx, maxy - miny)

    if scale > 0:
        poly = affinity.scale(poly, xfact=1/scale, yfact=1/scale, origin=(0,0))

    return poly


def normalized_shape_jaccard(poly_a, poly_b):
    try:
        a = normalize_shape(poly_a)
        b = normalize_shape(poly_b)

        inter = a.intersection(b).area
        union = a.area + b.area - inter

        if union <= 0:
            return 0.0

        return float(inter / union)

    except:
        return 0.0


qid = 40000
nid = gt_map[qid][0]

score = normalized_shape_jaccard(polygons[qid], polygons[nid])

print("Normalized Shape Jaccard:", score)

Normalized Shape Jaccard: 0.2811257880059238


In [40]:
# ==========================================
# CELL F4: Show Top-5 GT Neighbors with
# Normalized Shape Jaccard Scores
# ==========================================

qid = 40000   # change query if needed

print("Query:", qid)
print("-" * 60)

top5 = gt_map[qid][:5]

for rank, nid in enumerate(top5, start=1):
    score = normalized_shape_jaccard(polygons[qid], polygons[nid])

    print(
        f"Rank {rank:>2} | Neighbor {nid:>6} | "
        f"Normalized Shape Jaccard = {score:.4f}"
    )

Query: 40000
------------------------------------------------------------
Rank  1 | Neighbor  26769 | Normalized Shape Jaccard = 0.2811
Rank  2 | Neighbor  35012 | Normalized Shape Jaccard = 0.6211
Rank  3 | Neighbor  12304 | Normalized Shape Jaccard = 0.4354
Rank  4 | Neighbor  39137 | Normalized Shape Jaccard = 0.6054
Rank  5 | Neighbor  28042 | Normalized Shape Jaccard = 0.2366


In [41]:
# ==========================================
# CELL R1: Polygon -> 64x64 Raster Image
# ==========================================

from shapely.geometry import box
from shapely.prepared import prep

RASTER_SIZE = 64

def normalize_polygon_for_raster(poly):
    """
    Center polygon and scale into [-0.5, 0.5] box.
    """
    c = poly.centroid
    poly = affinity.translate(poly, xoff=-c.x, yoff=-c.y)

    minx, miny, maxx, maxy = poly.bounds
    scale = max(maxx - minx, maxy - miny)

    if scale > 0:
        poly = affinity.scale(poly, xfact=1/scale, yfact=1/scale, origin=(0, 0))

    return poly


def polygon_to_raster(poly, size=64):
    """
    Convert polygon into binary occupancy image.
    Output shape: (1, size, size)
    """
    poly = normalize_polygon_for_raster(poly)
    prepared = prep(poly)

    img = np.zeros((size, size), dtype=np.float32)

    cell_size = 1.0 / size
    start = -0.5

    for i in range(size):
        for j in range(size):
            minx = start + j * cell_size
            miny = start + i * cell_size
            maxx = minx + cell_size
            maxy = miny + cell_size

            cell = box(minx, miny, maxx, maxy)

            if prepared.intersects(cell):
                img[size - 1 - i, j] = 1.0

    return img[None, :, :]


# test one raster
rimg = polygon_to_raster(polygons[40000], RASTER_SIZE)

print("Raster shape:", rimg.shape)
print("Active cells:", rimg.sum())

Raster shape: (1, 64, 64)
Active cells: 1623.0


In [43]:
# ==========================================
# CELL R2P: 100-Core Parallel Raster Build
# ==========================================

from concurrent.futures import ProcessPoolExecutor, as_completed
import time
import numpy as np
from tqdm.auto import tqdm

RASTER_SIZE = 64
RASTER_PATH = OUT_DIR / f"raster_{len(polygons)}_{RASTER_SIZE}.npy"

def rasterize_range(args):
    start, end, size = args

    block = np.zeros((end - start, 1, size, size), dtype=np.float32)

    for local_i, poly_idx in enumerate(range(start, end)):
        try:
            block[local_i] = polygon_to_raster(polygons[poly_idx], size)
        except Exception:
            block[local_i] = 0.0

    return start, block


def build_rasters_100core(polygons, size=64, workers=80, chunk_size=100):
    n = len(polygons)

    tasks = []
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        tasks.append((start, end, size))

    XR = np.zeros((n, 1, size, size), dtype=np.float32)

    print("Total polygons:", n)
    print("Workers:", workers)
    print("Chunk size:", chunk_size)
    print("Total tasks:", len(tasks))

    t0 = time.time()

    with ProcessPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(rasterize_range, task) for task in tasks]

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Rasterizing"):
            start, block = fut.result()
            XR[start:start + len(block)] = block

    dt = time.time() - t0

    print("Done.")
    print("Time sec:", round(dt, 2))
    print("Time min:", round(dt / 60, 2))
    print("XR shape:", XR.shape)
    print("Memory MB:", round(XR.nbytes / 1024 / 1024, 2))

    return XR


# Run parallel rasterization
if RASTER_PATH.exists():
    print("Loading cached rasters...")
    XR = np.load(RASTER_PATH)
else:
    XR = build_rasters_100core(
        polygons,
        size=RASTER_SIZE,
        workers=80,       # try 64, 80, 96
        chunk_size=100     # try 100 or 200
    )

    np.save(RASTER_PATH, XR)
    print("Saved:", RASTER_PATH)

print("Final XR shape:", XR.shape)
print("Sample active cells:", XR[40000].sum())

Total polygons: 50000
Workers: 80
Chunk size: 100
Total tasks: 500


Rasterizing:   0%|          | 0/500 [00:00<?, ?it/s]

Done.
Time sec: 103.68
Time min: 1.73
XR shape: (50000, 1, 64, 64)
Memory MB: 781.25
Saved: /raid/ruban/hpmlproj/embedding/raster_50000_64.npy
Final XR shape: (50000, 1, 64, 64)
Sample active cells: 1623.0


In [44]:
# ==========================================
# CELL R3: Raster Pair Dataset
# ==========================================

class RasterPairDataset(Dataset):
    def __init__(self, XR, pairs):
        self.XR = XR
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        a, b, y = self.pairs[idx]

        xa = torch.tensor(self.XR[a], dtype=torch.float32)
        xb = torch.tensor(self.XR[b], dtype=torch.float32)
        y  = torch.tensor(y, dtype=torch.float32)

        return xa, xb, y


# use balanced_pairs from before
raster_ds = RasterPairDataset(XR, balanced_pairs)

raster_loader = DataLoader(
    raster_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

# test batch
xa, xb, y = next(iter(raster_loader))

print("xa:", xa.shape)
print("xb:", xb.shape)
print("y :", y.shape)

xa: torch.Size([256, 1, 64, 64])
xb: torch.Size([256, 1, 64, 64])
y : torch.Size([256])


In [45]:
# ==========================================
# CELL R4: Raster CNN Encoder
# ==========================================

class RasterCNNEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 -> 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 16 -> 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.fc = nn.Linear(256, embed_dim)

    def forward(self, x):
        feat = self.conv(x)
        feat = feat.view(feat.size(0), -1)

        emb = self.fc(feat)
        emb = F.normalize(emb, p=2, dim=1)

        return emb


raster_model = RasterCNNEncoder(embed_dim=EMBED_DIM).to(DEVICE)

with torch.no_grad():
    test_emb = raster_model(xa.to(DEVICE))

print("Embedding shape:", test_emb.shape)
print("First norm:", test_emb[0].norm().item())

Embedding shape: torch.Size([256, 128])
First norm: 1.0


In [46]:
# ==========================================
# CELL R5: Train Raster CNN
# ==========================================

optimizer = torch.optim.Adam(
    raster_model.parameters(),
    lr=2e-4,
    weight_decay=1e-5
)

criterion = nn.MSELoss()


def train_raster_epoch(model, loader):
    model.train()

    total_loss = 0.0
    count = 0

    pbar = tqdm(loader)

    for xa, xb, y in pbar:

        xa = xa.to(DEVICE, non_blocking=True)
        xb = xb.to(DEVICE, non_blocking=True)
        y  = y.to(DEVICE, non_blocking=True)

        ea = model(xa)
        eb = model(xb)

        pred = F.cosine_similarity(ea, eb)

        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

        avg = total_loss / count
        pbar.set_description(f"loss={avg:.4f}")

    return total_loss / count


history = []

for epoch in range(1, 6):
    t0 = time.time()

    loss_val = train_raster_epoch(raster_model, raster_loader)

    dt = time.time() - t0

    history.append(loss_val)

    print(f"Epoch {epoch} | Loss={loss_val:.4f} | Time={dt:.1f}s")

  0%|          | 0/1476 [00:00<?, ?it/s]

Epoch 1 | Loss=0.0792 | Time=48.6s


  0%|          | 0/1476 [00:00<?, ?it/s]

Epoch 2 | Loss=0.0751 | Time=51.2s


  0%|          | 0/1476 [00:00<?, ?it/s]

Epoch 3 | Loss=0.0738 | Time=52.9s


  0%|          | 0/1476 [00:00<?, ?it/s]

Epoch 4 | Loss=0.0730 | Time=49.0s


  0%|          | 0/1476 [00:00<?, ?it/s]

Epoch 5 | Loss=0.0723 | Time=49.6s


In [47]:
# ==========================================
# CELL R6: Generate Raster Embeddings
# ==========================================

def generate_raster_embeddings(model, XR, batch_size=512):
    model.eval()

    out = []

    with torch.no_grad():
        for i in tqdm(range(0, len(XR), batch_size), desc="Embedding"):
            xb = torch.tensor(XR[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            emb = model(xb)
            out.append(emb.cpu().numpy())

    return np.vstack(out).astype("float32")


raster_embeddings = generate_raster_embeddings(
    raster_model,
    XR,
    batch_size=512
)

print("Shape:", raster_embeddings.shape)
print("Norm:", np.linalg.norm(raster_embeddings[0]))

Embedding:   0%|          | 0/98 [00:00<?, ?it/s]

Shape: (50000, 128)
Norm: 1.0


In [48]:
# ==========================================
# CELL R7: FAISS Recall Evaluation
# ==========================================

import faiss
import numpy as np
from tqdm.auto import tqdm

# use raster embeddings
embeddings = raster_embeddings.astype("float32")

# normalized embeddings -> cosine via inner product
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS index size:", index.ntotal)


def recall_at_k(gt_map, query_ids, k=10):
    vals = []

    for qid in tqdm(query_ids, desc=f"Recall@{k}"):

        if qid >= len(embeddings):
            continue

        gt = gt_map.get(qid, [])
        if len(gt) == 0:
            continue

        gt_topk = set(gt[:k])

        qvec = embeddings[qid:qid+1]

        scores, idxs = index.search(qvec, k + 1)

        preds = [x for x in idxs[0] if x != qid][:k]

        hit = len(set(preds) & gt_topk)

        vals.append(hit / k)

    return float(np.mean(vals))


query_ids = sorted(gt_map.keys())

r10 = recall_at_k(gt_map, query_ids, k=10)
r50 = recall_at_k(gt_map, query_ids, k=50)

print("Recall@10 :", round(r10, 4))
print("Recall@50 :", round(r50, 4))

FAISS index size: 50000


Recall@10:   0%|          | 0/10000 [00:00<?, ?it/s]

Recall@50:   0%|          | 0/10000 [00:00<?, ?it/s]

Recall@10 : 0.0003
Recall@50 : 0.001


In [49]:
# ==========================================
# DEBUG 1: Embedding Collapse Check
# ==========================================

print("Mean std across dims:", raster_embeddings.std(axis=0).mean())

sims_to_first = raster_embeddings @ raster_embeddings[0]
print("Mean sim to first:", sims_to_first.mean())
print("Min sim to first :", sims_to_first.min())
print("Max sim to first :", sims_to_first.max())

# random pair similarities
idx = np.random.choice(len(raster_embeddings), size=5000, replace=False)
S = raster_embeddings[idx] @ raster_embeddings[idx].T

print("Random sim mean:", S[np.triu_indices_from(S, k=1)].mean())
print("Random sim std :", S[np.triu_indices_from(S, k=1)].std())

Mean std across dims: 0.05387929
Mean sim to first: 0.6457486
Min sim to first : 0.3889754
Max sim to first : 1.0000001
Random sim mean: 0.6255467
Random sim std : 0.07255226


In [50]:
# ==========================================
# CELL T1: Build GT Ranking Triplets
# ==========================================

def build_gt_rank_triplets(gt_map, db_size, top_pos=5, neg_per_pos=5):
    triplets = []
    all_ids = list(range(db_size))

    for qid, nbrs in tqdm(gt_map.items(), desc="Building GT triplets"):
        if qid >= db_size:
            continue

        positives = [nid for nid in nbrs[:top_pos] if nid < db_size]
        forbidden = set(nbrs) | {qid}

        if not positives:
            continue

        for pos in positives:
            for _ in range(neg_per_pos):
                neg = random.choice(all_ids)

                while neg in forbidden:
                    neg = random.choice(all_ids)

                triplets.append((qid, pos, neg))

    random.shuffle(triplets)
    return triplets


gt_triplets = build_gt_rank_triplets(
    gt_map=gt_map,
    db_size=len(XR),
    top_pos=5,
    neg_per_pos=5
)

print("Triplets:", len(gt_triplets))
print("Sample:", gt_triplets[:5])

Building GT triplets:   0%|          | 0/10000 [00:00<?, ?it/s]

Triplets: 230370
Sample: [(48661, 35430, 40629), (45715, 26077, 31441), (44757, 32016, 12231), (47992, 10489, 48069), (42463, 29983, 10207)]


In [51]:
# ==========================================
# CELL T2: Raster Triplet Dataset
# ==========================================

class RasterTripletDataset(Dataset):
    def __init__(self, XR, triplets):
        self.XR = XR
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a, p, n = self.triplets[idx]

        xa = torch.tensor(self.XR[a], dtype=torch.float32)
        xp = torch.tensor(self.XR[p], dtype=torch.float32)
        xn = torch.tensor(self.XR[n], dtype=torch.float32)

        return xa, xp, xn


triplet_ds = RasterTripletDataset(XR, gt_triplets)

triplet_loader = DataLoader(
    triplet_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

xa, xp, xn = next(iter(triplet_loader))

print("xa:", xa.shape)
print("xp:", xp.shape)
print("xn:", xn.shape)

xa: torch.Size([256, 1, 64, 64])
xp: torch.Size([256, 1, 64, 64])
xn: torch.Size([256, 1, 64, 64])


In [52]:
# ==========================================
# CELL T3: Train Retrieval Model
# ==========================================

# fresh model recommended
raster_model = RasterCNNEncoder(embed_dim=128).to(DEVICE)

optimizer = torch.optim.Adam(
    raster_model.parameters(),
    lr=2e-4,
    weight_decay=1e-5
)

MARGIN = 0.20


def train_triplet_epoch(model, loader):
    model.train()

    total_loss = 0.0
    count = 0

    pbar = tqdm(loader)

    for xa, xp, xn in pbar:

        xa = xa.to(DEVICE, non_blocking=True)
        xp = xp.to(DEVICE, non_blocking=True)
        xn = xn.to(DEVICE, non_blocking=True)

        ea = model(xa)
        ep = model(xp)
        en = model(xn)

        sim_pos = F.cosine_similarity(ea, ep)
        sim_neg = F.cosine_similarity(ea, en)

        loss = torch.relu(MARGIN - sim_pos + sim_neg).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

        avg = total_loss / count
        pbar.set_description(f"loss={avg:.4f}")

    return total_loss / count


history = []

for epoch in range(1, 6):
    t0 = time.time()

    loss_val = train_triplet_epoch(raster_model, triplet_loader)

    dt = time.time() - t0

    history.append(loss_val)

    print(f"Epoch {epoch} | Loss={loss_val:.4f} | Time={dt:.1f}s")

  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 1 | Loss=0.1997 | Time=38.5s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 2 | Loss=0.1990 | Time=41.8s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 3 | Loss=0.1977 | Time=39.5s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 4 | Loss=0.1956 | Time=38.2s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 5 | Loss=0.1917 | Time=38.8s


In [53]:
# ==========================================
# DEBUG: Raw Raster Jaccard Positive vs Negative
# ==========================================

def raster_jaccard(a, b):
    A = XR[a, 0] > 0
    B = XR[b, 0] > 0
    inter = np.logical_and(A, B).sum()
    union = np.logical_or(A, B).sum()
    return inter / union if union > 0 else 0.0


pos_scores = []
neg_scores = []

qids = random.sample(list(gt_map.keys()), 500)

for qid in qids:
    if qid >= len(XR):
        continue

    # top GT positives
    for nid in gt_map[qid][:5]:
        if nid < len(XR):
            pos_scores.append(raster_jaccard(qid, nid))

    # random negatives
    for _ in range(5):
        rid = random.randint(0, len(XR)-1)
        neg_scores.append(raster_jaccard(qid, rid))

print("Raw raster positive mean:", np.mean(pos_scores))
print("Raw raster negative mean:", np.mean(neg_scores))
print("Gap:", np.mean(pos_scores) - np.mean(neg_scores))

Raw raster positive mean: 0.4099996492494986
Raw raster negative mean: 0.41208434069811756
Gap: -0.002084691448618947


In [54]:
# ==========================================
# DEBUG: Model sim_pos vs sim_neg
# ==========================================

raster_model.eval()

sim_pos_all = []
sim_neg_all = []

with torch.no_grad():
    for xa, xp, xn in list(triplet_loader)[:20]:
        xa = xa.to(DEVICE)
        xp = xp.to(DEVICE)
        xn = xn.to(DEVICE)

        ea = raster_model(xa)
        ep = raster_model(xp)
        en = raster_model(xn)

        sim_pos = F.cosine_similarity(ea, ep)
        sim_neg = F.cosine_similarity(ea, en)

        sim_pos_all.extend(sim_pos.cpu().numpy())
        sim_neg_all.extend(sim_neg.cpu().numpy())

print("Model sim_pos mean:", np.mean(sim_pos_all))
print("Model sim_neg mean:", np.mean(sim_neg_all))
print("Gap:", np.mean(sim_pos_all) - np.mean(sim_neg_all))

Model sim_pos mean: 0.8955218
Model sim_neg mean: 0.88309145
Gap: 0.01243037


In [58]:
# ==========================================
# CELL W1: Inspect weighted encoding files
# ==========================================

import os, glob
from pathlib import Path

ENCODING_DIR = "/raid/ruban/encodings/pk-real50k0.002"  # change if needed

files = sorted(glob.glob(os.path.join(ENCODING_DIR, "real_*.txt")))

print("Files found:", len(files))
print("First 5 files:")
for f in files[:5]:
    print(f)

# peek one file
sample_file = files[0]

print("\nSample file:", sample_file)
with open(sample_file, "r") as f:
    for i in range(5):
        line = f.readline().strip()
        print(line[:300])

Files found: 80
First 5 files:
/raid/ruban/encodings/pk-real50k0.002/real_0.txt
/raid/ruban/encodings/pk-real50k0.002/real_10000.txt
/raid/ruban/encodings/pk-real50k0.002/real_10625.txt
/raid/ruban/encodings/pk-real50k0.002/real_11250.txt
/raid/ruban/encodings/pk-real50k0.002/real_11875.txt

Sample file: /raid/ruban/encodings/pk-real50k0.002/real_0.txt
0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 
0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 
0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0

In [59]:
# ==========================================
# CELL W2: Load real_*.txt Dense Encodings
# ==========================================

import os, glob
import numpy as np
from tqdm.auto import tqdm

ENC_DIR = "/raid/ruban/encodings/pk-real50k0.002"

files = sorted(glob.glob(os.path.join(ENC_DIR, "real_*.txt")))

blocks = []

for fp in tqdm(files, desc="Loading files"):
    arr = np.loadtxt(fp, dtype=np.float32)
    
    # if single row file
    if arr.ndim == 1:
        arr = arr[None, :]
        
    blocks.append(arr)

Xw = np.vstack(blocks).astype(np.float32)

print("Shape:", Xw.shape)
print("Memory MB:", round(Xw.nbytes / 1024 / 1024, 2))
print("Min:", Xw.min(), "Max:", Xw.max())
print("Mean nonzero per row:", (Xw > 0).sum(axis=1).mean())

Loading files:   0%|          | 0/80 [00:00<?, ?it/s]

Shape: (50000, 18382)
Memory MB: 3506.09
Min: 0.0 Max: 0.16067412
Mean nonzero per row: 4897.82258


In [60]:
# ==========================================
# CELL W3: Recall using raw weighted vectors
# ==========================================

import faiss
import numpy as np

# L2 normalize for cosine
Xn = Xw.copy()
norms = np.linalg.norm(Xn, axis=1, keepdims=True) + 1e-12
Xn = Xn / norms

dim = Xn.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(Xn)

print("Index size:", index.ntotal)

embeddings = Xn   # reuse eval code

Index size: 50000


In [62]:
# ==========================================
# CELL W4P: Fast Batched Recall Evaluation
# ==========================================

import numpy as np
from tqdm.auto import tqdm
import time

query_ids = np.array(sorted(gt_map.keys()), dtype=np.int64)

# keep only valid ids
query_ids = query_ids[query_ids < len(embeddings)]

KMAX = 50
BATCH = 1024

all_preds = {}

t0 = time.time()

for i in tqdm(range(0, len(query_ids), BATCH), desc="FAISS batches"):
    batch_ids = query_ids[i:i+BATCH]

    Q = embeddings[batch_ids]

    scores, idxs = index.search(Q, KMAX + 1)

    for row, qid in enumerate(batch_ids):
        pred = [x for x in idxs[row] if x != qid][:KMAX]
        all_preds[int(qid)] = pred

print("Search time sec:", round(time.time() - t0, 2))


def compute_recall(k):
    vals = []

    for qid in query_ids:
        gt = gt_map[int(qid)]
        gt_topk = set(gt[:k])

        pred = all_preds[int(qid)][:k]

        hit = len(set(pred) & gt_topk)

        vals.append(hit / k)

    return float(np.mean(vals))


r10 = compute_recall(10)
r50 = compute_recall(50)

print("Recall@10 :", round(r10, 4))
print("Recall@50 :", round(r50, 4))

FAISS batches:   0%|          | 0/10 [00:00<?, ?it/s]

Search time sec: 228.81
Recall@10 : 0.0002
Recall@50 : 0.0009


In [63]:
import os, re, glob

files = glob.glob("/raid/ruban/encodings/pk-real50k0.002/real_*.txt")

# current lexical sort
lex = sorted(files)

print("Lexical first 10:")
for f in lex[:10]:
    print(os.path.basename(f))

def get_id(fp):
    return int(re.search(r"real_(\d+)\.txt", fp).group(1))

num = sorted(files, key=get_id)

print("\nNumeric first 10:")
for f in num[:10]:
    print(os.path.basename(f))

Lexical first 10:
real_0.txt
real_10000.txt
real_10625.txt
real_11250.txt
real_11875.txt
real_1250.txt
real_12500.txt
real_13125.txt
real_13750.txt
real_14375.txt

Numeric first 10:
real_0.txt
real_625.txt
real_1250.txt
real_1875.txt
real_2500.txt
real_3125.txt
real_3750.txt
real_4375.txt
real_5000.txt
real_5625.txt


In [64]:
# ==========================================
# FIX LOAD ORDER
# ==========================================

import os, re, glob
import numpy as np
from tqdm.auto import tqdm

ENC_DIR = "/raid/ruban/encodings/pk-real50k0.002"

files = glob.glob(os.path.join(ENC_DIR, "real_*.txt"))

def get_id(fp):
    return int(re.search(r"real_(\d+)\.txt", fp).group(1))

files = sorted(files, key=get_id)

blocks = []

for fp in tqdm(files, desc="Loading"):
    arr = np.loadtxt(fp, dtype=np.float32)

    if arr.ndim == 1:
        arr = arr[None, :]

    blocks.append(arr)

Xw = np.vstack(blocks).astype(np.float32)

print("Shape:", Xw.shape)

Loading:   0%|          | 0/80 [00:00<?, ?it/s]

Shape: (50000, 18382)


In [65]:
# ==========================================
# FULL BASELINE EVAL (Correct Order)
# Normalize + FAISS + Batched Recall
# ==========================================

import os, re, glob, time
import numpy as np
import faiss
from tqdm.auto import tqdm

# -------------------------------------------------
# 1. LOAD FILES IN NUMERIC ORDER
# -------------------------------------------------

ENC_DIR = "/raid/ruban/encodings/pk-real50k0.002"

files = glob.glob(os.path.join(ENC_DIR, "real_*.txt"))

def get_id(fp):
    return int(re.search(r"real_(\d+)\.txt", fp).group(1))

files = sorted(files, key=get_id)

print("Files:", len(files))
print("First 10:")
for f in files[:10]:
    print(os.path.basename(f))

# -------------------------------------------------
# 2. LOAD MATRIX
# -------------------------------------------------

blocks = []

for fp in tqdm(files, desc="Loading vectors"):
    arr = np.loadtxt(fp, dtype=np.float32)

    if arr.ndim == 1:
        arr = arr[None, :]

    blocks.append(arr)

Xw = np.vstack(blocks).astype(np.float32)

print("\nLoaded shape:", Xw.shape)
print("Memory MB:", round(Xw.nbytes / 1024 / 1024, 2))

# -------------------------------------------------
# 3. L2 NORMALIZE FOR COSINE SEARCH
# -------------------------------------------------

norms = np.linalg.norm(Xw, axis=1, keepdims=True) + 1e-12
Xn = Xw / norms

print("Normalized.")

# -------------------------------------------------
# 4. BUILD FAISS INDEX
# -------------------------------------------------

dim = Xn.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(Xn)

print("FAISS index size:", index.ntotal)

# -------------------------------------------------
# 5. BATCH SEARCH ALL GT QUERIES
# -------------------------------------------------

query_ids = np.array(sorted(gt_map.keys()), dtype=np.int64)
query_ids = query_ids[query_ids < len(Xn)]

KMAX = 50
BATCH = 1024

all_preds = {}

t0 = time.time()

for i in tqdm(range(0, len(query_ids), BATCH), desc="FAISS search"):
    batch_ids = query_ids[i:i+BATCH]

    Q = Xn[batch_ids]

    scores, idxs = index.search(Q, KMAX + 1)

    for row, qid in enumerate(batch_ids):
        preds = [x for x in idxs[row] if x != qid][:KMAX]
        all_preds[int(qid)] = preds

print("Search time sec:", round(time.time() - t0, 2))

# -------------------------------------------------
# 6. COMPUTE RECALL
# -------------------------------------------------

def compute_recall(k):
    vals = []

    for qid in query_ids:
        gt = gt_map[int(qid)]
        gt_topk = set(gt[:k])

        pred = all_preds[int(qid)][:k]

        hit = len(set(pred) & gt_topk)

        vals.append(hit / k)

    return float(np.mean(vals))

r10 = compute_recall(10)
r50 = compute_recall(50)

print("\nRecall@10 :", round(r10, 4))
print("Recall@50 :", round(r50, 4))

Files: 80
First 10:
real_0.txt
real_625.txt
real_1250.txt
real_1875.txt
real_2500.txt
real_3125.txt
real_3750.txt
real_4375.txt
real_5000.txt
real_5625.txt


Loading vectors:   0%|          | 0/80 [00:00<?, ?it/s]


Loaded shape: (50000, 18382)
Memory MB: 3506.09
Normalized.
FAISS index size: 50000


FAISS search:   0%|          | 0/10 [00:00<?, ?it/s]

Search time sec: 211.31

Recall@10 : 0.5935
Recall@50 : 0.5885


In [66]:
# ==========================================
# CELL P1: PCA -> 128D -> Recall
# ==========================================

from sklearn.decomposition import PCA
import time
import numpy as np
import faiss
from tqdm.auto import tqdm

TARGET_DIM = 128

# ------------------------------------------
# 1. PCA FIT + TRANSFORM
# ------------------------------------------

t0 = time.time()

pca = PCA(n_components=TARGET_DIM, svd_solver="randomized", random_state=42)

Xp = pca.fit_transform(Xw).astype(np.float32)

print("PCA done in sec:", round(time.time() - t0, 2))
print("Shape:", Xp.shape)

# explained variance
print("Explained variance ratio:",
      round(pca.explained_variance_ratio_.sum(), 4))

# ------------------------------------------
# 2. Normalize for cosine
# ------------------------------------------

norms = np.linalg.norm(Xp, axis=1, keepdims=True) + 1e-12
Xp = Xp / norms

# ------------------------------------------
# 3. Build FAISS
# ------------------------------------------

index = faiss.IndexFlatIP(TARGET_DIM)
index.add(Xp)

print("FAISS index size:", index.ntotal)

# ------------------------------------------
# 4. Batched Search
# ------------------------------------------

query_ids = np.array(sorted(gt_map.keys()), dtype=np.int64)
query_ids = query_ids[query_ids < len(Xp)]

KMAX = 50
BATCH = 1024

all_preds = {}

t0 = time.time()

for i in tqdm(range(0, len(query_ids), BATCH), desc="FAISS search"):
    batch_ids = query_ids[i:i+BATCH]

    Q = Xp[batch_ids]

    scores, idxs = index.search(Q, KMAX + 1)

    for row, qid in enumerate(batch_ids):
        preds = [x for x in idxs[row] if x != qid][:KMAX]
        all_preds[int(qid)] = preds

print("Search time sec:", round(time.time() - t0, 2))

# ------------------------------------------
# 5. Recall
# ------------------------------------------

def compute_recall(k):
    vals = []

    for qid in query_ids:
        gt_topk = set(gt_map[int(qid)][:k])
        pred = all_preds[int(qid)][:k]

        hit = len(set(pred) & gt_topk)
        vals.append(hit / k)

    return float(np.mean(vals))

r10 = compute_recall(10)
r50 = compute_recall(50)

print("\nRecall@10 :", round(r10, 4))
print("Recall@50 :", round(r50, 4))

PCA done in sec: 26.57
Shape: (50000, 128)
Explained variance ratio: 1.0
FAISS index size: 50000


FAISS search:   0%|          | 0/10 [00:00<?, ?it/s]

Search time sec: 5.81

Recall@10 : 0.0127
Recall@50 : 0.0187


In [67]:
# ==========================================
# CELL N1: Weighted Vector -> 128D Embedding
# ==========================================

class WeightedMLPEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(512, embed_dim)
        )

    def forward(self, x):
        emb = self.net(x)
        emb = F.normalize(emb, p=2, dim=1)
        return emb


input_dim = Xw.shape[1]

w_model = WeightedMLPEncoder(
    input_dim=input_dim,
    embed_dim=128
).to(DEVICE)

# test
xb = torch.tensor(Xw[:256], dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    emb = w_model(xb)

print("Input dim:", input_dim)
print("Embedding shape:", emb.shape)
print("Norm:", emb[0].norm().item())

Input dim: 18382
Embedding shape: torch.Size([256, 128])
Norm: 1.0


In [68]:
# ==========================================
# CELL N2: Weighted Triplet Dataset
# ==========================================

class WeightedTripletDataset(Dataset):
    def __init__(self, Xw, triplets):
        self.Xw = Xw
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a, p, n = self.triplets[idx]

        xa = torch.tensor(self.Xw[a], dtype=torch.float32)
        xp = torch.tensor(self.Xw[p], dtype=torch.float32)
        xn = torch.tensor(self.Xw[n], dtype=torch.float32)

        return xa, xp, xn


w_triplet_ds = WeightedTripletDataset(Xw, gt_triplets)

w_loader = DataLoader(
    w_triplet_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

xa, xp, xn = next(iter(w_loader))

print("xa:", xa.shape)
print("xp:", xp.shape)
print("xn:", xn.shape)

xa: torch.Size([256, 18382])
xp: torch.Size([256, 18382])
xn: torch.Size([256, 18382])


In [69]:
# ==========================================
# CELL N3: Train Weighted Embedding Model
# ==========================================

# fresh model
w_model = WeightedMLPEncoder(
    input_dim=Xw.shape[1],
    embed_dim=128
).to(DEVICE)

optimizer = torch.optim.Adam(
    w_model.parameters(),
    lr=2e-4,
    weight_decay=1e-5
)

MARGIN = 0.20


def train_weighted_epoch(model, loader):
    model.train()

    total_loss = 0.0
    count = 0

    pbar = tqdm(loader)

    for xa, xp, xn in pbar:

        xa = xa.to(DEVICE, non_blocking=True)
        xp = xp.to(DEVICE, non_blocking=True)
        xn = xn.to(DEVICE, non_blocking=True)

        ea = model(xa)
        ep = model(xp)
        en = model(xn)

        sim_pos = F.cosine_similarity(ea, ep)
        sim_neg = F.cosine_similarity(ea, en)

        loss = torch.relu(MARGIN - sim_pos + sim_neg).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

        avg = total_loss / count
        pbar.set_description(f"loss={avg:.4f}")

    return total_loss / count


history = []

for epoch in range(1, 6):
    t0 = time.time()

    loss_val = train_weighted_epoch(w_model, w_loader)

    dt = time.time() - t0

    history.append(loss_val)

    print(f"Epoch {epoch} | Loss={loss_val:.4f} | Time={dt:.1f}s")

  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 1 | Loss=0.0725 | Time=66.9s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 2 | Loss=0.0339 | Time=69.5s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 3 | Loss=0.0285 | Time=58.5s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 4 | Loss=0.0215 | Time=59.5s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 5 | Loss=0.0207 | Time=57.2s


In [70]:
# ==========================================
# CELL N4: Generate Learned Embeddings
# ==========================================

def generate_weighted_embeddings(model, Xw, batch_size=512):
    model.eval()
    out = []

    with torch.no_grad():
        for i in tqdm(range(0, len(Xw), batch_size), desc="Embedding"):
            xb = torch.tensor(Xw[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            emb = model(xb)
            out.append(emb.cpu().numpy())

    return np.vstack(out).astype(np.float32)


learned_emb = generate_weighted_embeddings(
    w_model,
    Xw,
    batch_size=512
)

print("Shape:", learned_emb.shape)
print("Norm:", np.linalg.norm(learned_emb[0]))

Embedding:   0%|          | 0/98 [00:00<?, ?it/s]

Shape: (50000, 128)
Norm: 1.0


In [71]:
# ==========================================
# CELL N5: Recall Eval for learned_emb
# ==========================================

import faiss
import numpy as np
import time
from tqdm.auto import tqdm

# ------------------------------------------
# 1. Use learned embeddings
# ------------------------------------------

embeddings = learned_emb.astype(np.float32)

dim = embeddings.shape[1]

# embeddings already L2 normalized from model
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS index size:", index.ntotal)

# ------------------------------------------
# 2. Query IDs
# ------------------------------------------

query_ids = np.array(sorted(gt_map.keys()), dtype=np.int64)
query_ids = query_ids[query_ids < len(embeddings)]

KMAX = 50
BATCH = 1024

all_preds = {}

# ------------------------------------------
# 3. Batched Search
# ------------------------------------------

t0 = time.time()

for i in tqdm(range(0, len(query_ids), BATCH), desc="FAISS search"):
    batch_ids = query_ids[i:i+BATCH]

    Q = embeddings[batch_ids]

    scores, idxs = index.search(Q, KMAX + 1)

    for row, qid in enumerate(batch_ids):
        preds = [x for x in idxs[row] if x != qid][:KMAX]
        all_preds[int(qid)] = preds

search_sec = time.time() - t0

print("Search time sec:", round(search_sec, 2))

# ------------------------------------------
# 4. Recall
# ------------------------------------------

def compute_recall(k):
    vals = []

    for qid in query_ids:
        gt_topk = set(gt_map[int(qid)][:k])
        pred = all_preds[int(qid)][:k]

        hit = len(set(pred) & gt_topk)
        vals.append(hit / k)

    return float(np.mean(vals))

r10 = compute_recall(10)
r50 = compute_recall(50)

print("\nRecall@10 :", round(r10, 4))
print("Recall@50 :", round(r50, 4))

FAISS index size: 50000


FAISS search:   0%|          | 0/10 [00:00<?, ?it/s]

Search time sec: 4.3

Recall@10 : 0.0022
Recall@50 : 0.0046


In [72]:
# average similarity gap on learned_emb
def avg_gap(E, gt_map, n=500):
    pos, neg = [], []
    qids = random.sample(list(gt_map.keys()), n)

    for q in qids:
        v = E[q]
        for p in gt_map[q][:5]:
            pos.append(float(v @ E[p]))
        for _ in range(5):
            r = random.randint(0, len(E)-1)
            neg.append(float(v @ E[r]))

    print("pos:", np.mean(pos))
    print("neg:", np.mean(neg))
    print("gap:", np.mean(pos)-np.mean(neg))

avg_gap(learned_emb, gt_map)

pos: 0.9999996024840357
neg: 0.9998642810583115
gap: 0.0001353214257242552


In [73]:
# ==========================================
# CELL D1: Anti-Collapse Weighted MLP
# ==========================================

class WeightedMLPEncoderNoNorm(nn.Module):
    def __init__(self, input_dim, embed_dim=512):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(4096, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(1024, embed_dim)
        )

    def forward(self, x):
        return self.net(x)   # no normalize here


EMBED_DIM2 = 512

w_model2 = WeightedMLPEncoderNoNorm(
    input_dim=Xw.shape[1],
    embed_dim=EMBED_DIM2
).to(DEVICE)

print(w_model2)

WeightedMLPEncoderNoNorm(
  (net): Sequential(
    (0): Linear(in_features=18382, out_features=4096, bias=True)
    (1): BatchNorm1d(4096, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=4096, out_features=1024, bias=True)
    (5): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=1024, out_features=512, bias=True)
  )
)


In [74]:
# ==========================================
# CELL D2: Anti-Collapse Training
# ==========================================

optimizer = torch.optim.Adam(
    w_model2.parameters(),
    lr=2e-4,
    weight_decay=1e-5
)

MARGIN = 0.50
LAMBDA_VAR = 0.05


def variance_penalty(z):
    """
    Encourage spread across batch + dimensions.
    Low std => penalty high
    """
    std = torch.std(z, dim=0) + 1e-4
    return torch.mean(torch.relu(1.0 - std))


def train_epoch_anticollapse(model, loader):
    model.train()

    total = 0.0
    count = 0

    pbar = tqdm(loader)

    for xa, xp, xn in pbar:

        xa = xa.to(DEVICE, non_blocking=True)
        xp = xp.to(DEVICE, non_blocking=True)
        xn = xn.to(DEVICE, non_blocking=True)

        ea = model(xa)
        ep = model(xp)
        en = model(xn)

        # cosine uses normalized vectors internally
        sim_pos = F.cosine_similarity(ea, ep)
        sim_neg = F.cosine_similarity(ea, en)

        triplet = torch.relu(MARGIN - sim_pos + sim_neg).mean()

        var_loss = (
            variance_penalty(ea) +
            variance_penalty(ep) +
            variance_penalty(en)
        ) / 3.0

        loss = triplet + LAMBDA_VAR * var_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()
        count += 1

        pbar.set_description(f"loss={total/count:.4f}")

    return total / count


history = []

for epoch in range(1, 6):
    t0 = time.time()

    loss_val = train_epoch_anticollapse(w_model2, w_loader)

    dt = time.time() - t0

    history.append(loss_val)

    print(f"Epoch {epoch} | Loss={loss_val:.4f} | Time={dt:.1f}s")

  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 1 | Loss=0.1989 | Time=73.8s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 2 | Loss=0.1429 | Time=70.6s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 3 | Loss=0.1120 | Time=73.5s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 4 | Loss=0.0989 | Time=63.8s


  0%|          | 0/900 [00:00<?, ?it/s]

Epoch 5 | Loss=0.0861 | Time=65.2s


In [75]:
# ==========================================
# CELL D3: Generate Anti-Collapse Embeddings
# ==========================================

def generate_emb(model, Xw, batch=512):
    model.eval()
    out = []

    with torch.no_grad():
        for i in tqdm(range(0, len(Xw), batch), desc="Embedding"):
            xb = torch.tensor(Xw[i:i+batch], dtype=torch.float32).to(DEVICE)
            z = model(xb)

            # normalize only for retrieval
            z = F.normalize(z, p=2, dim=1)

            out.append(z.cpu().numpy())

    return np.vstack(out).astype(np.float32)


emb2 = generate_emb(w_model2, Xw)

print("Shape:", emb2.shape)
print("Norm:", np.linalg.norm(emb2[0]))


# quick sanity gap
def avg_gap(E, gt_map, n=500):
    pos, neg = [], []
    qids = random.sample(list(gt_map.keys()), n)

    for q in qids:
        v = E[q]

        for p in gt_map[q][:5]:
            pos.append(float(v @ E[p]))

        for _ in range(5):
            r = random.randint(0, len(E)-1)
            neg.append(float(v @ E[r]))

    print("pos:", np.mean(pos))
    print("neg:", np.mean(neg))
    print("gap:", np.mean(pos)-np.mean(neg))

avg_gap(emb2, gt_map)

Embedding:   0%|          | 0/98 [00:00<?, ?it/s]

Shape: (50000, 512)
Norm: 1.0
pos: 0.9999999263525112
neg: 0.999992958521843
gap: 6.967830668225261e-06


In [77]:
def eval_embeddings(E):
    E = np.asarray(E, dtype=np.float32, order="C")

    norms = np.linalg.norm(E, axis=1, keepdims=True) + 1e-12
    E = E / norms

    E = np.ascontiguousarray(E, dtype=np.float32)

    index = faiss.IndexFlatIP(E.shape[1])
    index.add(E)

    KMAX = 50
    BATCH = 1024
    all_preds = {}

    t0 = time.time()

    for i in range(0, len(query_ids), BATCH):
        batch_ids = query_ids[i:i+BATCH]
        Q = E[batch_ids]

        scores, idxs = index.search(Q, KMAX + 1)

        for row, qid in enumerate(batch_ids):
            preds = [x for x in idxs[row] if x != qid][:KMAX]
            all_preds[int(qid)] = preds

    search_sec = time.time() - t0

    def rec(k):
        vals = []
        for qid in query_ids:
            gt_topk = set(gt_map[int(qid)][:k])
            pred = all_preds[int(qid)][:k]
            vals.append(len(set(pred) & gt_topk) / k)
        return float(np.mean(vals))

    return rec(10), rec(50), search_sec

In [78]:
results = []

for dim in DIMS:
    print(f"\n=== Testing {dim}D ===")

    t0 = time.time()

    rp = SparseRandomProjection(
        n_components=dim,
        dense_output=True,
        random_state=42
    )

    Xrp = rp.fit_transform(Xw)

    proj_sec = time.time() - t0

    r10, r50, search_sec = eval_embeddings(Xrp)

    results.append((dim, proj_sec, search_sec, r10, r50))

    print("Projection sec:", round(proj_sec, 2))
    print("Recall@10:", round(r10, 4))
    print("Recall@50:", round(r50, 4))
    print("Search sec:", round(search_sec, 2))

print("\nFINAL RESULTS")
for row in results:
    print(row)


=== Testing 512D ===
Projection sec: 3.61
Recall@10: 0.5586
Recall@50: 0.5575
Search sec: 8.91

=== Testing 1024D ===
Projection sec: 4.62
Recall@10: 0.5742
Recall@50: 0.5724
Search sec: 14.95

=== Testing 2048D ===
Projection sec: 6.91
Recall@10: 0.5834
Recall@50: 0.5802
Search sec: 26.26

FINAL RESULTS
(512, 3.60758900642395, 8.905286312103271, 0.5585899999999999, 0.557492)
(1024, 4.6235575675964355, 14.949026584625244, 0.57419, 0.57237)
(2048, 6.907886981964111, 26.260345935821533, 0.5834299999999999, 0.580188)
